## Core Concepts
### Trace: A complete execution of an operation

Represents one request or workflow
Contains one or more spans
Has a root span

### Span: A single operation within a trace

Has a start and end time
Contains inputs and outputs
Has metadata (model, tokens,latecy, etc.)
Can have parent-child relationships
Span Attributes: Additional metadata

Model name
Token counts
Temperature
Custom attributes
Span Types
MLflow defines standard span types:

CHAIN: A sequence of operations
LLM: Language model call
RETRIEVER: Document retrieval
EMBEDDING: Text embedding
TOOL: Tool/function execution
AGENT: Agent reasoning
PARSER: Output parsing or generic intermediate parsing of an outcome
Hierarchy Example
TRACE (root)
│
└─ SPAN: Agent Executor (AGENT)
   │
   ├─ SPAN: Planning Step (LLM)
   │  └─ attributes: {model: gpt-5, tokens: 50}
   │
   ├─ SPAN: Tool Execution (TOOL)
   │  └─ attributes: {tool: search, query: "..."}
   │
   └─ SPAN: Final Response (LLM)
      └─ attributes: {model: gpt-5, tokens: 150}


In [1]:
import mlflow
from dotenv import load_dotenv
from utils.clnt_utils import is_databricks_ai_gateway_client, get_databricks_ai_gateway_client, get_openai_client, get_ai_gateway_model_names

# Load environment
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("http://localhost:5000")

use_databricks_provider = is_databricks_ai_gateway_client()
if use_databricks_provider:
    client = get_databricks_ai_gateway_client()
    model_name = get_ai_gateway_model_names()[0]
else:
    # Initialize OpenAI
    client = get_openai_client()
    model_name = "gpt-5-mini"

print("✅ Environment configured: using", "Databricks" if use_databricks_provider else "OpenAI", "client")
print(f"   MLflow version: {mlflow.__version__}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")
print(f"   Using model: {model_name}")

✅ Environment configured: using OpenAI client
   MLflow version: 3.11.1
   Tracking URI: http://localhost:5000
   Using model: gpt-5-mini


In [2]:
import sys, mlflow
print(sys.executable)
print(mlflow.__version__)

/Users/ksulahian/ML_journey/learning_materials/llmops/.venv/bin/python3
3.11.1


In [4]:
# Create experiment for tracing examples
mlflow.set_experiment("06-tracing-introduction")

# Without tracing - basic call
print("\n📝 Making LLM call WITHOUT tracing...\n")


📝 Making LLM call WITHOUT tracing...



In [6]:


prompt = "Explain what distributed tracing is in one sentence."
# response = client.chat.completions.create(
#     model=model_name,
#     messages=[{"role": "user", "content": prompt}],
#     temperature=1.0,
#     max_completion_tokens=1000
# )

print(f"Response: {response.choices[0].message.content}")
print("\n❌ What we DON'T know:")
print("   - Exact timing of the call")
print("   - Detailed request/response structure")
print("   - Easy way to correlate with other operations")
print("   - Visual representation of execution")

# check the trace in the UI
print("\n🔍 View trace in MLflow UI: http://localhost:5000")
print("   Navigate to: Traces tab")

NameError: name 'response' is not defined

In [3]:
# Enable OpenAI autologging - THIS IS THE MAGIC LINE!
mlflow.openai.autolog()

print("✅ OpenAI autologging enabled")
print("   All OpenAI API calls will now be automatically traced!")

✅ OpenAI autologging enabled
   All OpenAI API calls will now be automatically traced!


In [7]:
print("\n🔍 Making LLM call WITH tracing...\n")

response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": prompt}],
    temperature=1.0,
    max_completion_tokens=1000
)

print(f"Response: {response.choices[0].message.content}")
print("\n✅ What we NOW know:")
print("   ✓ Complete request details (model, messages, parameters)")
print("   ✓ Response content and metadata")
print("   ✓ Token usage (prompt, completion, total)")
print("   ✓ Timing information (latency)")
print("   ✓ All captured automatically!")
print("\n🔗 View trace in MLflow UI: http://localhost:5000")
print("   Navigate to: Traces tab")


🔍 Making LLM call WITH tracing...

Response: Distributed tracing is a technique for following a single request as it travels across multiple services in a distributed system by recording timed spans and metadata so you can reconstruct the end-to-end execution, measure latency, and diagnose errors.

✅ What we NOW know:
   ✓ Complete request details (model, messages, parameters)
   ✓ Response content and metadata
   ✓ Token usage (prompt, completion, total)
   ✓ Timing information (latency)
   ✓ All captured automatically!

🔗 View trace in MLflow UI: http://localhost:5000
   Navigate to: Traces tab


Trace(trace_id=tr-f7892c242e4b1064c182b313d42d12e9)